# t-SNE of Real vs Generated Audio Features

Produces the paper's t-SNE figure of real vs generated audio features. Run it
from the repository root. On the MSP-PODCAST evaluation split it:

1. extracts the **real** audio features (`feat_audio`, the Wav2Vec2 output that
   `AudioEnc.forward` returns as `rep_real`);
2. **generates** one feature per real example with the InfoGAN generator
   `model.gan.G`, conditioned on the example's emotion label (one-hot). By
   default the continuous slot of the noise is a random Gaussian vector
   (`AudioEnc.compute_noise` with `t=None`), so **no text encoder is needed**;
   set `condition="text"` to condition on the matched text features instead
   (loads the 7B text backbone);
3. fits **one** t-SNE on the concatenation of generated + real features, so both
   sources share the same 2-D coordinates;
4. draws three figures — generated only, real only, and the overlay of both
   (color = emotion class, marker = source) — and saves each as PDF + PNG under
   `fig_dir`.

Requirements: a checkpoint containing the trained InfoGAN
(`checkpoints/audio_infogan.pt` or `checkpoints/audio_fusion.pt`) and
`matplotlib` / `scikit-learn`. The fusion stage does not update the generator or
the Wav2Vec2 encoder (see the optimizers in `audio_script.py`), so both
checkpoints give the same features.

In [ ]:
import os
import warnings
from contextlib import nullcontext

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.manifold import TSNE
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import Wav2Vec2Config, Wav2Vec2Processor

import common
from src.datacollator import DataCollatorCTCWithPadding
from src.dataset import Dataset_aug

warnings.filterwarnings("ignore")

## Arguments

Edit this cell and run everything below. Data/model defaults mirror
`validate_generation.ipynb`.

- `n_samples` — number of evaluation examples used (default: the first 2500 of
  the split; `None` = whole split, which is slow for t-SNE and very packed).
  `random_subset=True` draws them at random (seeded) instead of taking the
  first `n_samples`.
- `condition` — `"rand"` (label + random Gaussian vector in the continuous
  slot, no text model) or `"text"` (label + matched text features; loads the
  Stage-1 text encoder from `text_ckpt`).
- `tsne_perplexity` — scikit-learn's default (30, with PCA initialisation).

In [ ]:
from types import SimpleNamespace

# Which GPU(s) to use. Must be set before the first CUDA call in this
# process (i.e. before the torch.cuda.is_available() below) to take effect.
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

args = SimpleNamespace(
    # data
    dataset="msp-podcast",       # "iemocap" | "msp-improv" | "msp-podcast"
    data_root="../MSP-PODCAST-Publish-1.10/",  # None -> common.DEFAULT_DATA_ROOT[dataset]
    split="test",                # "test" | "dev"
    fold=0,                      # IEMOCAP / MSP-IMPROV only
    refresh_cache=False,         # re-parse instead of loading cache/
    n_samples=2500,              # None -> whole split; default: first 2500 examples
    random_subset=False,         # True -> seeded random subset instead of the first n
    # models
    ckpt="checkpoints/audio_fusion.pt",   # checkpoint with the InfoGAN weights to visualise
    condition="rand",            # "rand" (random continuous slot) | "text" (matched text)
    text_ckpt="checkpoints/text.pt",      # Stage-1 text encoder; only used when condition="text"
    text_backbone=common.DEFAULT_TEXT_BACKBONE,
    audio_backbone=common.DEFAULT_AUDIO_BACKBONE,
    # run
    batch_size=16,               # inference only; raise if memory allows
    seed=230,                    # seeds z, the random continuous vectors, subsampling and t-SNE
    # t-SNE / figures
    tsne_perplexity=30,
    fig_dir="figures",
)

if args.data_root is None:
    args.data_root = common.DEFAULT_DATA_ROOT[args.dataset]
assert args.condition in ("rand", "text"), f"unknown condition '{args.condition}'"
assert os.path.exists(args.ckpt), f"ckpt '{args.ckpt}' not found"
if args.condition == "text":
    assert os.path.exists(args.text_ckpt), f"text_ckpt '{args.text_ckpt}' not found"

torch.manual_seed(args.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Data

By default the first 2500 examples of the split are visualised; `n_samples` /
`random_subset` change that (see *Arguments*).

In [ ]:
dataset = common.build_dataset(args.dataset, args.data_root, args.refresh_cache)
num_labels = dataset.num_labels
_, dev_examples, test_examples = common.resolve_splits(dataset, args.fold)
examples = test_examples if args.split == "test" else dev_examples

id2label = {}
for name, idx in dataset.label_list.items():
    id2label.setdefault(idx, name)
class_names = [id2label.get(i, str(i)) for i in range(num_labels)]

n_total = len(examples)
if args.n_samples is not None and args.n_samples < n_total:
    if args.random_subset:
        rng = np.random.default_rng(args.seed)
        keep = np.sort(rng.choice(n_total, args.n_samples, replace=False))
        examples = [examples[i] for i in keep]
    else:
        examples = examples[:args.n_samples]
print(f"{args.dataset} {args.split}: using {len(examples)} of {n_total} examples, "
      f"classes = {class_names}")

## Models

The audio encoder whose `gan.G` is the generator under test. The frozen Stage-1
text encoder (7B backbone) is loaded only for `condition="text"`.

In [ ]:
config_a = Wav2Vec2Config.from_pretrained(args.audio_backbone, num_labels=num_labels)
config_a.proj_size = config_a.hidden_size
processor = Wav2Vec2Processor.from_pretrained(args.audio_backbone)

model = common.build_audio_encoder(config_a, num_labels, args.audio_backbone, ckpt=args.ckpt)
model.to(device)
model.eval()

eval_set = Dataset_aug(list(examples))
if args.condition == "text":
    txt, _, tokenizer = common.build_text_encoder(
        args.text_backbone, num_labels, device, freeze=True, ckpt=args.text_ckpt)
    txt.eval()
    eval_set.set_tokenizer(tokenizer)
else:
    txt = None

loader = DataLoader(eval_set, batch_size=args.batch_size, shuffle=False,
                    collate_fn=DataCollatorCTCWithPadding(processor=processor, padding=True),
                    num_workers=0)

## Run — extract real features, generate one counterpart per example

Per batch: the real features come straight from the Wav2Vec2 forward (the same
tensor `AudioEnc.forward` returns as `rep_real`, without paying for the D/G/Q
loss passes). Generation calls `compute_noise` with the batch's one-hot labels
and either `t=None` (random Gaussian continuous slot) or the matched text
features, then runs the generator. Every generated feature therefore
carries the label of its real counterpart, which is what the per-class colouring
below relies on.

The per-sample features are kept on the CPU in float32; t-SNE and the plots
work off the concatenated tensors.

In [ ]:
amp = (torch.autocast(device.type, dtype=torch.float16)
       if device.type == "cuda" else nullcontext())

feats_real, feats_gen, targets = [], [], []

with torch.no_grad():
    for data in tqdm(loader, unit="batch", dynamic_ncols=True):
        c = data["labels"].view(-1, num_labels).float()  # one-hot labels (CPU)
        with amp:
            feat_audio = model.audio_wav2vec2(
                input_values=data["input_values"].to(device))[0]
            t = txt(data, device)[0] if txt is not None else None
            noise = model.compute_noise(c.shape[0], model.gan.z_dim,
                                        model.gan.disc_dim, model.gan.cont_dim,
                                        c=c, t=t)
            fake = model.gan.G(noise.to(device))

        feats_real.append(feat_audio.view(c.shape[0], -1).float().cpu())
        feats_gen.append(fake.float().cpu())
        targets.extend(torch.argmax(c, dim=-1).tolist())

feats_real = torch.cat(feats_real).numpy()  # (N, 1024) real audio features
feats_gen = torch.cat(feats_gen).numpy()    # (N, 1024) G(z, label, rand | text)
y = np.array(targets)
print(f"done: {len(y)} examples per source; per class: "
      f"{np.bincount(y, minlength=num_labels).tolist()}")

## t-SNE — shared 2-D embedding

One t-SNE is fitted on the **concatenation** of the generated and the real
features (generated first) and split back, so the two sources share the same
coordinates and the three figures below are directly comparable. scikit-learn's
defaults are spelled out (perplexity 30, PCA initialisation, automatic learning
rate) and seeded.

In [ ]:
stacked = np.concatenate([feats_gen, feats_real])
tsne = TSNE(n_components=2, perplexity=args.tsne_perplexity, init="pca",
            learning_rate="auto", random_state=args.seed, verbose=1)
proj = tsne.fit_transform(stacked)
proj_gen, proj_real = proj[:len(y)], proj[len(y):]

## Figures

Three figures: one colour per emotion class, `.` for generated and `+` for real
features, alpha 0.5.

1. generated features only, coloured by (conditioning) class;
2. real features only, coloured by class;
3. overlay of both on the shared coordinates (`<class>,gen` / `<class>,real`).

Each figure is saved to `fig_dir` as PDF (vector — use this in the paper) and PNG,
tagged with the conditioning mode and the checkpoint name.

In [ ]:
plt.rcParams["pdf.fonttype"] = 42  # embed editable (TrueType) text in the PDFs

CLASS_COLORS = ["red", "blue", "green", "orange"]           # one colour per class
MARKERS = {"gen": ".", "real": "+"}                           # source -> marker
# legend names as printed in the paper; falls back to the loader's label names
LEGEND_NAMES = {"N": "neu", "H": "hap", "A": "ang", "S": "sad"}
legend_names = [LEGEND_NAMES.get(c, c) for c in class_names]
if num_labels > len(CLASS_COLORS):
    CLASS_COLORS = [plt.cm.tab10(i) for i in range(num_labels)]

ckpt_tag = os.path.splitext(os.path.basename(args.ckpt))[0]
os.makedirs(args.fig_dir, exist_ok=True)


def save_fig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(args.fig_dir, f"{name}.{ext}"),
                    dpi=200, bbox_inches="tight")
    print(f"saved {args.fig_dir}/{name}.pdf and .png")


def draw_source(ax, emb, source, with_source_in_label=False):
    for i, name in enumerate(legend_names):
        m = y == i
        label = f"{name},{source}" if with_source_in_label else name
        ax.scatter(emb[m, 0], emb[m, 1], c=CLASS_COLORS[i],
                   marker=MARKERS[source], label=label, alpha=0.5)


# 1 + 2: one source per figure
for source, emb in (("gen", proj_gen), ("real", proj_real)):
    fig, ax = plt.subplots(figsize=(8, 8))
    draw_source(ax, emb, source)
    ax.legend(fontsize="small", markerscale=2)
    ax.set_title(f"t-SNE of {'generated' if source == 'gen' else 'real'} audio features "
                 f"({args.dataset} {args.split}, {len(y)} examples)")
    save_fig(fig, f"tsne_{source}_{args.condition}_{ckpt_tag}")
    plt.show()

# 3: overlay — per class, generated ('.') then real ('+') on the shared coordinates
fig, ax = plt.subplots(figsize=(8, 8))
for i, name in enumerate(legend_names):
    m = y == i
    for source, emb in (("gen", proj_gen), ("real", proj_real)):
        ax.scatter(emb[m, 0], emb[m, 1], c=CLASS_COLORS[i], marker=MARKERS[source],
                   label=f"{name},{source}", alpha=0.5)
ax.legend(fontsize="small", markerscale=2)
ax.set_title(f"t-SNE of generated (.) vs real (+) audio features "
             f"({args.dataset} {args.split}, {len(y)} examples/source)")
save_fig(fig, f"tsne_overlay_{args.condition}_{ckpt_tag}")
plt.show()